# A4 — Penetration Depth of Concrete Target
### Li & Chen (2003), *Dimensionless formulae for penetration depth of concrete target impacted by a non-deformable projectile*, IJIE 28, 93–116

🔗 [Engineering Tools — endofwave.github.io/engineering-tools/](https://endofwave.github.io/engineering-tools/)

---

This notebook computes the **penetration depth** of a rigid projectile into a semi-infinite concrete target using the closed-form dimensionless formulae of Li & Chen (2003).  
The pipeline is based on cavity expansion theory (Forrestal 1994) and uses two dimensionless numbers — the **impact function I** and the **geometry function N** — to predict X/d.  
Valid for: rigid (non-deformable) steel projectiles, normal incidence, semi-infinite concrete targets, impact velocity V₀ ≲ 800 m/s.  
Every intermediate value is printed at each node — no black box.

**Pipeline nodes:**
| Node | Description |
|:-----|:------------|
| 0 | Validity pre-check |
| 1 | Nose geometry → N*, k |
| 2 | Target resistance → S |
| 3 | Dimensionless numbers → I, N, Φ_J |
| 4 | Regime selection (shallow / deep) |
| 5 | Penetration depth → X/d |
| 6 | Dimensional output → X [mm] |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# INPUT PARAMETERS — Modify these values for your case
# Default values: Forrestal et al. (1994), Table 3, shot 14
# ============================================================

# Projectile
M          = 0.906        # Mass [kg]
d          = 0.0269       # Shank diameter [m]
V0         = 277.0        # Impact velocity [m/s]
nose_type  = "ogive"      # Options: "flat", "ogive", "conical", "spherical"
psi        = 2.0          # Nose parameter: CRH (R/d) for ogive,
                          #                 H/d  for conical,
                          #                 r/d  for spherical

# Target (concrete)
fc         = 35.2e6       # Unconfined compressive strength [Pa]
rho_c      = 2370.0       # Density [kg/m³]

# Options
S_correlation         = "simplified"  # "original"   → Forrestal 1994, Eq. (12)
                                      # "simplified" → Li & Chen 2003,  Eq. (21)
apply_shallow_correction = False      # Eq. (27) correction for X/d < 0.5

In [ ]:
# ============================================================
# NODE 0 — VALIDITY PRE-CHECK
# ============================================================

warnings_list = []

# Rigid projectile velocity limit
if V0 > 800:
    warnings_list.append(
        f"⚠️  V0 = {V0:.0f} m/s > 800 m/s: projectile deformation/erosion may be significant"
    )

# Nose type validity
if nose_type not in ["flat", "ogive", "conical", "spherical"]:
    warnings_list.append(
        f"⚠️  nose_type = '{nose_type}' is not recognised. Use: flat, ogive, conical, spherical."
    )

# Nose parameter lower bound
if nose_type == "ogive" and psi < 0.5:
    warnings_list.append(f"⚠️  CRH ψ = {psi} < 0.5 is not physically valid for an ogive nose")
if nose_type == "spherical" and psi < 0.5:
    warnings_list.append(f"⚠️  r/d ψ = {psi} < 0.5 is not valid for a spherical nose")

print("=" * 60)
print("NODE 0 — Validity Pre-Check")
print("=" * 60)
if warnings_list:
    for w in warnings_list:
        print(w)
else:
    print("✅  All basic checks passed")
print()
print(f"  V0          = {V0:.1f} m/s  (model limit ≲ 800 m/s for hardened steel)")
print(f"  Nose type   = {nose_type}")
print(f"  Incidence   = normal (oblique impact not covered by this model)")
print(f"  Target      = semi-infinite (verify thickness ≥ 3X after calculation)")

In [ ]:
# ============================================================
# NODE 1 — NOSE GEOMETRY
# ============================================================

def compute_nose_factor(nose_type, psi):
    """Compute N* — nose shape factor, Eqs. (3a)–(3d) of Li & Chen (2003).

    N* ranges from 0 (ideal sharp) to 1 (flat-nosed).
    A lower N* means lower target resistance during penetration.
    """
    if nose_type == "flat":
        return 1.0
    elif nose_type == "ogive":
        return 1 / (3 * psi) - 1 / (24 * psi**2)
    elif nose_type == "conical":
        return 1 / (1 + 4 * psi**2)
    elif nose_type == "spherical":
        return 1 - 1 / (8 * psi**2)
    else:
        raise ValueError(f"Unknown nose_type: '{nose_type}'")


def compute_nose_height(nose_type, psi):
    """Compute H/d — dimensionless nose height (crater depth factor).

    Used to compute k = 0.707 + H/d, Eq. (25).
    """
    if nose_type == "flat":
        return 0.0
    elif nose_type == "ogive":
        return np.sqrt(psi - 0.25)
    elif nose_type == "conical":
        return psi
    elif nose_type == "spherical":
        return psi - np.sqrt(psi**2 - 0.25)
    else:
        raise ValueError(f"Unknown nose_type: '{nose_type}'")


N_star    = compute_nose_factor(nose_type, psi)
H_over_d  = compute_nose_height(nose_type, psi)
k         = 0.707 + H_over_d   # Eq. (25): crater + nose-tip correction

print("=" * 60)
print("NODE 1 — Nose Geometry")
print("=" * 60)
print(f"  Nose type   = {nose_type}")
print(f"  ψ           = {psi}")
print(f"  N*          = {N_star:.6f}   (nose shape factor, Eq. 3b)")
print(f"  H/d         = {H_over_d:.6f}   (dimensionless nose height)")
print(f"  k           = 0.707 + {H_over_d:.4f} = {k:.6f}   (Eq. 25)")

In [ ]:
# ============================================================
# NODE 2 — TARGET RESISTANCE (S)
# ============================================================
# S is an empirical dimensionless constant that scales target
# resistance with compressive strength.
# Two correlations are available:
#   Forrestal 1994, Eq. (12): S = 82.6 · fc^(-0.544)
#   Li & Chen 2003,  Eq. (21): S = 72.0 · fc^(-0.5)   ← simplified form
# fc in MPa for both.
# ============================================================

fc_MPa = fc / 1e6   # Convert Pa → MPa

S_original   = 82.6 * fc_MPa**(-0.544)   # Forrestal 1994
S_simplified = 72.0 * fc_MPa**(-0.5)     # Li & Chen 2003

if S_correlation == "original":
    S = S_original
elif S_correlation == "simplified":
    S = S_simplified
else:
    raise ValueError(f"Unknown S_correlation: '{S_correlation}'")

print("=" * 60)
print("NODE 2 — Target Resistance")
print("=" * 60)
print(f"  fc              = {fc_MPa:.2f} MPa")
print(f"  S (original,   Eq.12) = {S_original:.4f}   [Forrestal 1994]")
print(f"  S (simplified, Eq.21) = {S_simplified:.4f}   [Li & Chen 2003]")
print(f"  S selected      = {S:.4f}   ({S_correlation})") 

In [ ]:
# ============================================================
# NODE 3 — DIMENSIONLESS NUMBERS
# ============================================================
# I*  — impact factor      (projectile energy / target resistance), Eq. (5)
# λ   — mass ratio         (projectile mass / displaced concrete mass), Eq. (6)
# I   — impact function    = I* / S,   Eq. (14)
# N   — geometry function  = λ / N*,  Eq. (13)
# Φ_J — Johnson damage number = I* / λ, Eq. (7)
# ============================================================

I_star = M * V0**2 / (d**3 * fc)   # Eq. (5)
lam    = M / (rho_c * d**3)        # Eq. (6)
I      = I_star / S                 # Eq. (14)
N      = lam / N_star               # Eq. (13)
Phi_J  = I_star / lam               # Eq. (7)

print("=" * 60)
print("NODE 3 — Dimensionless Numbers")
print("=" * 60)
print(f"  I* = M·V₀² / (d³·fc)       = {I_star:.4f}   (impact factor, Eq. 5)")
print(f"  λ  = M / (ρc·d³)            = {lam:.4f}   (mass ratio,   Eq. 6)")
print(f"  I  = I* / S                  = {I:.4f}   (impact function, Eq. 14)")
print(f"  N  = λ / N*                  = {N:.2f}   (geometry function, Eq. 13)")
print(f"  I/N                          = {I/N:.6f}")
print(f"  Φ_J = I* / λ                = {Phi_J:.4f}   (Johnson damage number, Eq. 7)")

In [ ]:
# ============================================================
# NODE 4 — REGIME SELECTION
# ============================================================
# Two regimes based on whether the projectile penetrates beyond
# the crater zone:
#
#   I ≤ πk/4  →  shallow penetration  (projectile stops within crater)
#   I >  πk/4  →  deep penetration    (projectile enters tunnel zone)
#
# Eq. (16): threshold at I = πk/4
# ============================================================

threshold = np.pi * k / 4   # Eq. (16)

if I > threshold:
    regime       = "deep"
    regime_label = "Deep penetration (crater + tunnel)"
    eq_label     = "Eq. (15b)"
else:
    regime       = "shallow"
    regime_label = "Shallow penetration (crater only)"
    eq_label     = "Eq. (15a)"

print("=" * 60)
print("NODE 4 — Regime Selection")
print("=" * 60)
print(f"  I      = {I:.4f}")
print(f"  πk/4   = π·{k:.4f}/4 = {threshold:.4f}")
print(f"  I {'>' if regime == 'deep' else '≤'} πk/4  →  {regime_label}")
print(f"  Formula to apply: {eq_label}")

In [ ]:
# ============================================================
# NODE 5 — PENETRATION DEPTH
# ============================================================

def eq15a(I, N, k):
    """Shallow penetration — Eq. (15a), valid for X/d ≤ k.

    Derived from energy balance during crater formation:
        X/d = sqrt( (4k·I/π) · (1 + kπ/(4N)) / (1 + I/N) )
    """
    return np.sqrt(
        (4 * k * I / np.pi) * (1 + k * np.pi / (4 * N)) / (1 + I / N)
    )


def eq15b(I, N, k):
    """Deep penetration — Eq. (15b), valid for X/d > k.

    Derived from equation of motion + cavity expansion in tunnel zone:
        X/d = (2/π)·N·ln[ (1 + I/N) / (1 + kπ/(4N)) ] + k
    """
    return (2 / np.pi) * N * np.log(
        (1 + I / N) / (1 + k * np.pi / (4 * N))
    ) + k


if regime == "shallow":
    Xd = eq15a(I, N, k)
else:
    Xd = eq15b(I, N, k)

# Optional shallow-range correction (Eq. 27), valid only for X/d < 0.5
Xd_corrected = None
if apply_shallow_correction and Xd < 0.5:
    Xd_corrected = 1.628 * Xd**2.789   # Eq. (27)

print("=" * 60)
print(f"NODE 5 — Penetration Depth  [{regime} regime]")
print("=" * 60)
print(f"  Formula used: {eq_label}")
print(f"  X/d = {Xd:.6f}")
if Xd_corrected is not None:
    print(f"  X/d (Eq. 27 correction, X/d < 0.5) = {Xd_corrected:.6f}")
elif apply_shallow_correction and Xd >= 0.5:
    print(f"  Eq. (27) correction not applied (X/d = {Xd:.4f} ≥ 0.5)")

In [ ]:
# ============================================================
# NODE 6 — DIMENSIONAL OUTPUT
# ============================================================

X    = Xd * d            # [m]
X_mm = X * 1000          # [mm]

if Xd_corrected is not None:
    X_corr    = Xd_corrected * d
    X_corr_mm = X_corr * 1000

print("=" * 60)
print("NODE 6 — Dimensional Output")
print("=" * 60)
print(f"  Regime:         {regime_label}")
print(f"  X/d             = {Xd:.4f}")
print(f"  X               = {X_mm:.2f} mm  ({X:.5f} m)")
if Xd_corrected is not None:
    print(f"  X/d (corr.)     = {Xd_corrected:.4f}")
    print(f"  X (corr.)       = {X_corr_mm:.2f} mm")

print()
print("  ── Intermediate values ──")
print(f"  N*   = {N_star:.6f}     k    = {k:.6f}")
print(f"  S    = {S:.4f}         I    = {I:.4f}")
print(f"  λ    = {lam:.4f}         N    = {N:.2f}")
print(f"  I*   = {I_star:.4f}         I/N  = {I/N:.6f}")
print(f"  Φ_J  = {Phi_J:.4f}")
print()
print(f"  ── Semi-infinite check ──")
print(f"  Target thickness should be ≥ {3*X_mm:.0f} mm  (= 3·X = 3·{X_mm:.1f} mm)")

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"{'Parameter':<28} {'Value':>14}   {'Unit':<8}")
print("-" * 60)
print(f"{'M  (mass)':<28} {M:>14.3f}   {'kg':<8}")
print(f"{'d  (diameter)':<28} {d*1000:>14.2f}   {'mm':<8}")
print(f"{'V0 (impact velocity)':<28} {V0:>14.1f}   {'m/s':<8}")
print(f"{'nose type':<28} {nose_type:>14}")
print(f"{'ψ  (nose parameter)':<28} {psi:>14.2f}")
print(f"{'fc (compressive strength)':<28} {fc_MPa:>14.2f}   {'MPa':<8}")
print(f"{'ρc (concrete density)':<28} {rho_c:>14.0f}   {'kg/m³':<8}")
print("-" * 60)
print(f"{'N* (nose shape factor)':<28} {N_star:>14.6f}")
print(f"{'k  (crater depth factor)':<28} {k:>14.6f}")
print(f"{'S  (target resistance)':<28} {S:>14.4f}")
print(f"{'I* (impact factor)':<28} {I_star:>14.4f}")
print(f"{'λ  (mass ratio)':<28} {lam:>14.4f}")
print(f"{'I  (impact function)':<28} {I:>14.4f}")
print(f"{'N  (geometry function)':<28} {N:>14.2f}")
print(f"{'I/N':<28} {I/N:>14.6f}")
print(f"{'Φ_J (Johnson number)':<28} {Phi_J:>14.4f}")
print(f"{'Regime':<28} {regime:>14}")
print("-" * 60)
print(f"{'X/d':<28} {Xd:>14.4f}")
print(f"{'X':<28} {X_mm:>14.2f}   {'mm':<8}")
print("=" * 60)

In [ ]:
# ============================================================
# PARAMETRIC PLOT 1 — X/d vs I  (reproduces Fig. 3 of the paper)
# Shows penetration depth as a function of impact function I
# for various geometry functions N.
# The current case is highlighted in red.
# ============================================================

I_range   = np.linspace(0.1, 200, 800)
N_values  = [20, 100, 300, 500, 1000, 1500]
k_plot    = k    # use the k from the current case

fig, ax = plt.subplots(figsize=(10, 7))

for N_val in N_values:
    Xd_vals = np.zeros_like(I_range)
    thr     = np.pi * k_plot / 4
    for i, I_val in enumerate(I_range):
        if I_val <= thr:
            Xd_vals[i] = eq15a(I_val, N_val, k_plot)
        else:
            Xd_vals[i] = eq15b(I_val, N_val, k_plot)
    ax.plot(I_range, Xd_vals, lw=1.5, label=f"N = {N_val}")

# Current case marker
ax.plot(I, Xd, 'ro', markersize=10, zorder=5,
        label=f"Current case  I = {I:.1f},  N = {N:.0f},  X/d = {Xd:.2f}")

ax.set_xlabel("Impact function  I", fontsize=12)
ax.set_ylabel("X/d", fontsize=12)
ax.set_title("Dimensionless penetration depth vs impact function\n"
             "(Li & Chen 2003, Fig. 3)", fontsize=13)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 200)
ax.set_ylim(0, 120)

plt.tight_layout()
plt.savefig("Xd_vs_I.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Xd_vs_I.png")

In [ ]:
# ============================================================
# PARAMETRIC PLOT 2 — X/d vs N  (reproduces Fig. 4 of the paper)
# Shows penetration depth as a function of geometry function N
# for various impact functions I.
# The current case is highlighted in red.
# ============================================================

N_range  = np.linspace(1, 1500, 800)
I_values = [10, 20, 40, 70, 100, 150, 200]

fig, ax = plt.subplots(figsize=(10, 7))

for I_val in I_values:
    Xd_vals = np.zeros_like(N_range)
    for i, N_val in enumerate(N_range):
        thr = np.pi * k_plot / 4
        if I_val <= thr:
            Xd_vals[i] = eq15a(I_val, N_val, k_plot)
        else:
            Xd_vals[i] = eq15b(I_val, N_val, k_plot)
    ax.plot(N_range, Xd_vals, lw=1.5, label=f"I = {I_val}")

# Current case marker
ax.plot(N, Xd, 'ro', markersize=10, zorder=5,
        label=f"Current case  I = {I:.1f},  N = {N:.0f},  X/d = {Xd:.2f}")

ax.set_xlabel("Geometry function  N", fontsize=12)
ax.set_ylabel("X/d", fontsize=12)
ax.set_title("Dimensionless penetration depth vs geometry function\n"
             "(Li & Chen 2003, Fig. 4)", fontsize=13)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1500)
ax.set_ylim(0, 120)

plt.tight_layout()
plt.savefig("Xd_vs_N.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Xd_vs_N.png")

In [ ]:
# ============================================================
# PARAMETRIC PLOT 3 — S vs fc  (reproduces Fig. 1 of the paper)
# Compares the two S correlations and shows the test data cloud.
# The current case is highlighted.
# ============================================================

fc_range      = np.linspace(10, 100, 400)   # MPa
S_orig_range  = 82.6 * fc_range**(-0.544)
S_simpl_range = 72.0 * fc_range**(-0.5)

# Representative test data from Li & Chen (2003) Fig. 1
fc_data = np.array([13.5, 23.0, 35.2, 62.8, 97.0])
S_data  = np.array([21.0, 15.2, 12.0,  9.1,  7.0])

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(fc_range, S_orig_range,  'b-',  lw=2,
        label=r"$S = 82.6\,f_c^{-0.544}$ — Forrestal 1994  [Eq. 12]")
ax.plot(fc_range, S_simpl_range, 'r--', lw=2,
        label=r"$S = 72.0\,f_c^{-0.5}$  — Li & Chen 2003  [Eq. 21]")
ax.plot(fc_data, S_data, 'ko', markersize=7, zorder=4, label="Test data (representative)")
ax.plot(fc_MPa, S, 'rs', markersize=12, zorder=5,
        label=f"Current case  (fc = {fc_MPa:.1f} MPa, S = {S:.2f})")

ax.set_xlabel(r"$f_c$ [MPa]", fontsize=12)
ax.set_ylabel("S", fontsize=12)
ax.set_title(r"Empirical constant $S$ vs compressive strength $f_c$"
             "\n(Li & Chen 2003, Fig. 1)", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 105)
ax.set_ylim(0, 25)

plt.tight_layout()
plt.savefig("S_vs_fc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: S_vs_fc.png")